In [50]:
import numpy as np
import pandas as pd
from tensorflow.keras import Sequential
from tensorflow import keras
from tensorflow.keras.models import model_from_json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [52]:
# Features
FEATURE_COLUMNS = [
    "forks_count",
    "subscribers_count",
    "open_issues_count",
    "size",
    "network_count",
    "has_wiki",
    "has_pages",
    "has_issues",
    "topics_count",
    "age_days",
    "language_encoded"
]
TARGET = "stargazers_count"


In [53]:
# load the dataset
df = pd.read_csv("github-repository-data.csv")
df.head()
df["language_encoded"] = pd.factorize(df["language"])[0]
pd.factorize(df["language"])[0]
# Log-transform target
df["log_stars"] = np.log1p(df["stargazers_count"])

X = df[FEATURE_COLUMNS].values
y = df["log_stars"].values

print(X.shape, y.shape)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaled the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

(1000, 11) (1000,)


In [57]:
# model
output_bias = keras.initializers.Constant(y_train.mean())

model = Sequential([
    keras.layers.Input(shape=(len(FEATURE_COLUMNS),)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, bias_initializer=output_bias)  # Output layer for regression
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

model.summary()


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_18 (Dense)                │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,881 (11.25 KB)

 Trainable params: 2,881 (11.25 KB)

 Non-trainable params: 0 (0.00 B)

In [58]:
# train the model
history = model.fit(
    X_train_scaled, 
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,     # 10% of train for val curve
    callbacks=[
        keras.callbacks.EarlyStopping(
            patience=10, restore_best_weights=True
        )
    ]
)


Epoch 1/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.2935 - mae: 0.3975 - val_loss: 0.1947 - val_mae: 0.3237
Epoch 2/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1767 - mae: 0.3252 - val_loss: 0.1977 - val_mae: 0.3107
Epoch 3/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1568 - mae: 0.3069 - val_loss: 0.1762 - val_mae: 0.2985
Epoch 4/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1387 - mae: 0.2902 - val_loss: 0.1576 - val_mae: 0.2841
Epoch 5/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1318 - mae: 0.2839 - val_loss: 0.1542 - val_mae: 0.2786
Epoch 6/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1313 - mae: 0.2777 - val_loss: 0.1527 - val_mae: 0.2777
Epoch 7/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1268 - mae: 0.2778 - val_loss: 0.1297 - val_mae: 0.2684
Epoch 8/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1303 - mae: 0.2689 - val_loss: 0.1308 - val_mae: 0.2649
Epoch 9/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.120

In [59]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
import numpy as np

def evaluate_model(model, X_test, y_test_log, model_name="Model"):
    """
    Evaluate a regression model predicting log(stars).
    
    Args:
        model      : trained model with a .predict() method
        X_test     : scaled test features
        y_test_log : true values in log scale (log1p transformed)
        model_name : label shown in the printed report
    
    Returns:
        dict with r2, mae, mape, accuracy
    """
    # Prediction
    y_prediction_log = model.predict(X_test).flatten()

    # Metrics on log scale
    r2 = r2_score(y_test_log, y_prediction_log)

    # Convert back to real star counts
    y_prediction_stars = np.expm1(y_prediction_log)
    y_test_stars = np.expm1(y_test_log)

    # Metrics on real scale
    mae  = mean_absolute_error(y_test_stars, y_prediction_stars)
    mape = mean_absolute_percentage_error(y_test_stars + 1, y_prediction_stars + 1)
    accuracy = max(0.0, 1.0 - mape)

    # Report
    print(f"\n{'='*45}")
    print(f"  {model_name}")
    print(f"{'='*45}")
    print(f"  R²               : {r2:.4f}")
    print(f"  MAE (stars)      : {mae:>12,.0f}")
    print(f"  MAPE             : {mape * 100:.2f}%")
    print(f"  Accuracy (1-MAPE): {accuracy * 100:.2f}%")
    print(f"{'='*45}")

    return {"model": model_name, "r2": r2, "mae": mae, "mape": mape, "accuracy": accuracy}


evaluate_model(model, X_test_scaled, y_test, model_name="Neural Network")


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 

  Neural Network
  R²               : 0.4889
  MAE (stars)      :       15,969
  MAPE             : 25.05%
  Accuracy (1-MAPE): 74.95%


{'model': 'Neural Network',
 'r2': 0.4888743492081038,
 'mae': 15968.671249999998,
 'mape': 0.2504657760572117,
 'accuracy': 0.7495342239427882}